# InduSense — Sprint 1 : intégration de la couche Bronze (journal de bord)

**Compétences** : C1 (jeu de données pertinent et cohérent), C2 (risques éthiques et réglementaires),
C3 (préparation et intégrité des données).

Ce notebook retrace **la méthode suivie pour intégrer les fichiers de `datas/` dans le schéma
`bronze` de PostgreSQL**, étape par étape. Comme le journal d'audit, il ne contient **pas** la logique
de traitement : elle vit dans `src/indusense/layers/` (testée, pilotée par `configs/`). Ici on
exécute chaque brique, on **regarde ce qu'elle produit** et on vérifie les garanties en base.

Références :
- consignes du formateur : `01-INGESTION_PROCESS.md` (contrat Bronze v2), `02-GUIDE_INGESTION_DONNEES.md`
  (méthode en 8 étapes), `guide_ingestion_medallion_indusense.md` (architecture), `13_RGPD_7_principes_Indusense.md` ;
- documentation du dépôt : `docs/08_bronze_contrat_v2.md`, `docs/07_guide_couches_bronze_silver_gold.md`.

### Le plan suit la méthode en 8 étapes du guide 02

| Étape (guide 02 §4) | Ce que fait le code | Module | Section |
|---|---|---|---|
| 0. Écrire le contrat avant de lire | contrats YAML versionnés | `configs/contracts/*.yaml`, `contracts/model.py` | 2 |
| 1. Inspecter la source | empreinte des octets, encodage, colonnes | `io/readers.py`, `io/files.py` | 3 |
| 2. Confronter déclaration et réalité | contrat vs fichier réel, erreur localisée | `layers/checks.py` | 4 |
| 3. Stocker brut, sans perte | `x_raw` + `x` typé + colonnes techniques | `layers/bronze.py`, `layers/sqltypes.py` | 5 |
| 4. Enregistrer le lot | seuil, transaction, registre des lots | `layers/stage.py`, `layers/database.py` | 6 |
| 5. Documenter | dictionnaire de données + `transformations` | `layers/dictionary.py` | 7 |
| 6. Tester les invariants | volume, fidélité, complétude, unicité | `tests/test_layers.py` + section 8 | 8 |
| 7. Sceller, rendre rejouable | clé primaire + `ON CONFLICT DO NOTHING` | `layers/database.py` | 9 |

Puis : le volet **réglementaire** (section 10) et **ce qu'il manque** par rapport aux consignes (section 11).

> Prérequis : `docker compose up -d db admin` et un fichier `.env` (copie de `.env.sample`).

## 0. Mise en place

On charge la configuration (`configs/pipeline.yaml`), les réglages des couches (section `layers:`)
et les variables du `.env` (connexion à la base). Le mot de passe n'est jamais affiché.

In [1]:
import logging
from importlib.metadata import version
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from sqlalchemy import text
from sqlalchemy.dialects import postgresql

from indusense.config import all_sources, load_settings
from indusense.contracts.model import load_contracts, structure_errors
from indusense.layers.anonymize import columns_to_pseudonymize, undecided_pii_columns
from indusense.layers.bronze import BatchMeta, build_bronze_frame, physical_columns, row_hash
from indusense.layers.checks import bronze_issues
from indusense.layers.database import EnvSettings, list_tables, make_engine
from indusense.layers.dictionary import silver_operations, transformations
from indusense.layers.migrate import schema_status
from indusense.layers.settings import LayerSettings
from indusense.layers.stage import _types, run_layers   # _types : type de chaque colonne (contrat, sinon inféré)
from indusense.pipeline import load_and_profile

logging.basicConfig(level=logging.INFO, format="%(levelname)-7s | %(message)s", force=True)
pd.set_option("display.max_columns", 50, "display.width", 200, "display.max_colwidth", 80)


def project_root(start: Path = Path.cwd()) -> Path:
    # Remonte l'arborescence jusqu'au dossier qui contient configs/pipeline.yaml.
    for p in [start, *start.parents]:
        if (p / "configs" / "pipeline.yaml").exists():
            return p
    raise FileNotFoundError("configs/pipeline.yaml introuvable")


ROOT = project_root()
settings = load_settings(ROOT / "configs" / "pipeline.yaml")
cfg = LayerSettings.from_raw(settings.raw, settings.root)
env = EnvSettings(_env_file=ROOT / ".env")     # le notebook tourne dans notebooks/ : chemin explicite
engine = make_engine(env)

In [2]:
print("Racine du projet      :", ROOT)
print("Config                :", settings.config_path.relative_to(ROOT), "| SHA-256", settings.config_sha256[:16], "…")
print("Schémas               :", cfg.bronze_schema, "/", cfg.silver_schema, "/", cfg.gold_schema)
print("Suffixe valeur brute  :", cfg.raw_suffix)
print("Registre des lots     :", f"{cfg.bronze_schema}.{cfg.registry_table}")
print("Seuil erreurs de cast :", f"{cfg.max_cast_error_ratio:.2%}")
print("Base                  :", f"{env.db_user}@{env.db_host}:{env.db_port}/{env.db_name}",
      "| mot de passe :", "renseigné" if env.db_password else "ABSENT")
with engine.connect() as conn:
    print("PostgreSQL            :", conn.execute(text("SHOW server_version")).scalar_one())

Racine du projet      : C:\dev\00-Fomation_IA\TP1\indusense
Config                : configs\pipeline.yaml | SHA-256 04ff2854441a201f …
Schémas               : bronze / silver / gold
Suffixe valeur brute  : _raw
Registre des lots     : bronze.ingestion_batches_log
Seuil erreurs de cast : 0.00%
Base                  : indusense_user@127.0.0.1:5434/indusense_db | mot de passe : renseigné
PostgreSQL            : 15.1 (Debian 15.1-1.pgdg110+1)


## 1. Les librairies utilisées, et pourquoi

| Librairie | Rôle dans le Bronze | Pourquoi elle (et pas autre chose) |
|---|---|---|
| **pandas** | lire les CSV et construire la table Bronze en mémoire | lecture en **texte brut** (`dtype=str`, `keep_default_na=False`, `na_filter=False`) : aucune conversion silencieuse, `"NA"` ou `"0012"` restent tels quels |
| **sqlglot** | lire le dump `.sql` | vrai **parseur SQL** : `CREATE TABLE` donne les colonnes déclarées, `INSERT` les colonnes réellement fournies. Le dump n'est **jamais exécuté**, et pas de regex (le guide 01 cite une regex qui perdait 98,5 % des lignes) |
| **SQLAlchemy 2** | décrire les tables, piloter les transactions | types SQL portables (`Text`, `BigInteger`, `Uuid`, `JSONB`…), clé primaire déclarée, `engine.begin()` = une transaction, `insert().on_conflict_do_nothing().returning()` pour l'idempotence, `inspect()` pour comparer la structure existante |
| **Alembic** | migrations de schéma du Bronze | structure générée depuis les contrats et les fichiers, **relue** puis appliquée (`indusense db revision` → `db upgrade`) : jamais de `CREATE TABLE` implicite (guide 01, étape 4) |
| **psycopg 3** | pilote PostgreSQL | pilote officiel moderne, utilisé par SQLAlchemy (`postgresql+psycopg://`) |
| **pydantic-settings** | lire le `.env` | les **secrets** (mot de passe, sel) restent hors du dépôt ; variables typées et validées |
| **PyYAML** | lire `pipeline.yaml` et les contrats | configuration déclarative : aucun nom de fichier ni de colonne dans le code |
| `hashlib` (stdlib) | SHA-256 du fichier et de chaque ligne | empreintes standard, calculées sur les **octets** pour le fichier |
| `uuid` (stdlib) | identifiant de lot `_batch_id` | unique sans coordination |
| `json` (stdlib) | empreinte de ligne canonique, `_cast_errors` | liste JSON : `["a,b","c"]` ≠ `["a","b,c"]` (une concaténation les confondrait) |
| `difflib` (stdlib) | « Vouliez-vous dire 'duration_hours' ? » | suggestion dans le message d'erreur localisé |
| `csv.Sniffer` (stdlib) | détection du séparateur | quand la config laisse `separator: auto` |
| `hmac` (stdlib) | pseudonymisation (au **Silver**) | HMAC-SHA256 = façon standard d'ajouter un sel secret |
| **PostgreSQL 15** + **pgAdmin** (Docker) | stockage des couches | `JSONB`, `TIMESTAMPTZ`, `ON CONFLICT`, schémas `bronze/silver/gold` ; base locale sur `127.0.0.1` |
| **uv**, **pytest** | dépendances, tests | environnement reproductible (`uv.lock`), invariants testés |

In [3]:
for pkg in ["pandas", "sqlglot", "sqlalchemy", "psycopg", "pydantic-settings", "pyyaml", "pytest"]:
    try:
        print(f"{pkg:<18} {version(pkg)}")
    except Exception:
        print(f"{pkg:<18} (non installé)")

pandas             3.1.1
sqlglot            30.33.0
sqlalchemy         2.1.9
psycopg            3.3.9
pydantic-settings  2.16.3
pyyaml             6.0.4
pytest             9.2.0


## 2. Étape 0 — Le contrat, écrit avant de lire

Un contrat (`configs/contracts/<id>.yaml`) déclare **l'attendu** : colonnes, type logique, type SQL
déclaré, clé primaire, colonnes en plus autorisées ou non, données personnelles (`pii`) et décision
de protection (`privacy`). Il est rattaché au fichier par son identifiant (nom sans extension),
grâce à la découverte automatique (écart assumé avec le `source.path` du formateur, cf. `docs/08`).

In [4]:
contracts = load_contracts(settings.contracts_dir)
rows = []
for cid, c in contracts.items():
    cols = c["schema"]["columns"]
    rows.append({"contrat": cid, "version": c["contract"].get("version"), "statut": c["contract"].get("status"),
                 "colonnes": len(cols), "clé primaire": c["schema"].get("primary_key"),
                 "colonnes en plus": c["schema"].get("allow_extra_columns", False),
                 "pii": [x["name"] for x in cols if x.get("pii")],
                 "erreurs de structure": structure_errors(c)})
pd.DataFrame(rows)

,contrat,version,statut,colonnes,clé primaire,colonnes en plus,pii,erreurs de structure
0,machine,2.2.0,active,11,[machine_code],False,[],[]
1,maintenance,0.5.0,draft,11,[maintenance_id],False,[],[]
2,releves_incidents,1.1.0,active,18,[incident_id],False,"[operator_name, operator_badge, comment]",[]
3,telemetry,0.3.0,draft,7,"[machine_id, timestamp]",False,[],[]


## 3. Étape 1 — Inspecter la source

Trois mesures qui ne coûtent presque rien (guide 02 §4, étape 1) : **l'empreinte des octets**
(identité du fichier), **l'encodage réel**, **les colonnes réellement présentes**. Pour un dump SQL,
on distingue en plus les colonnes **déclarées** au `CREATE TABLE` et celles **fournies** par un `INSERT`.

In [5]:
specs = all_sources(settings)
loaded = load_and_profile(specs, settings, contracts)

rows = []
for ls in loaded:
    s = ls.src
    for ds in s.datasets:
        rows.append({"fichier": s.spec.path.name, "jeu": ds.id, "format": s.spec.resolved_format,
                     "encodage": s.charset, "séparateur": s.separator, "taille": s.size_label,
                     "sha256 (octets)": s.sha256[:16] + "…", "lignes": len(ds.df), "colonnes": ds.df.shape[1],
                     "déclarées non fournies": ([c for c in ds.df.columns if c not in ds.provided_columns]
                                                if ds.provided_columns is not None else "—")})
pd.DataFrame(rows)

INFO    | lecture de machine.sql


INFO    | lecture de releves_incidents.csv


INFO    | lecture de telemetry.csv


,fichier,jeu,format,encodage,séparateur,taille,sha256 (octets),lignes,colonnes,déclarées non fournies
0,machine.sql,machine,sql,UTF-8,NaN,22.41 Ko,9eb877ae806515e8…,15,11,"[is_active, created_at, updated_at]"
1,machine.sql,maintenance,sql,UTF-8,NaN,22.41 Ko,9eb877ae806515e8…,115,11,"[created_at, updated_at]"
2,releves_incidents.csv,releves_incidents,csv,UTF-8,",",134.64 Ko,be7204b3320a39fa…,1245,18,—
3,telemetry.csv,telemetry,csv,ASCII (compatible UTF-8),",",7.94 Mo,68ec8dd09b691e68…,135626,7,—


## 4. Étape 2 — Confronter le contrat au fichier réel

C'est l'étape que la plupart des pipelines sautent (guide 02). `bronze_issues` renvoie la **liste
complète** des problèmes, chacun avec la colonne et la source. **Une seule erreur suffit : le jeu est
refusé et rien n'est écrit.**

| Contrôle | Bloquant ? |
|---|---|
| aucun contrat pour ce jeu | oui |
| contrat mal formé (`contract check`) | oui |
| colonne déclarée absente du fichier (avec suggestion) | oui |
| colonne du fichier absente du contrat | oui, sauf `allow_extra_columns: true` (alors avertissement) |
| colonne d'un dump SQL au `CREATE TABLE` mais fournie par aucun `INSERT` | oui, sauf champ de traçabilité (`traceability: true`) : avertissement, chargée vide |
| aucune ligne à ingérer | oui |
| collision de noms (`x` / `x_raw` / colonnes techniques) ou nom > 63 caractères | oui |

In [6]:
accepted, refused = [], {}
for ls in loaded:
    for ds, profile in zip(ls.src.datasets, ls.profiles):
        label = ls.src.spec.path.name + (f":{ds.table}" if ds.table else "")
        errors, warnings = bronze_issues(ds, contracts.get(ds.id), label, cfg)
        print(f"■ {ds.id:<20} {'REFUSÉ' if errors else 'accepté'}")
        for e in errors:
            print("    ERREUR        :", e)
        for w in warnings:
            print("    avertissement :", w)
        if errors:
            refused[ds.id] = errors
        else:
            accepted.append((ls, ds, profile))
print(f"\n{len(accepted)} jeu(x) accepté(s), {len(refused)} refusé(s) : {list(refused)}")

■ machine              accepté
    avertissement : 'is_active' (traçabilité) : aucune ligne de machine.sql:machine ne la fournit ; chargée vide (NULL) au Bronze, non reprise au Silver
    avertissement : 'created_at' (traçabilité) : aucune ligne de machine.sql:machine ne la fournit ; chargée vide (NULL) au Bronze, non reprise au Silver
    avertissement : 'updated_at' (traçabilité) : aucune ligne de machine.sql:machine ne la fournit ; chargée vide (NULL) au Bronze, non reprise au Silver
■ maintenance          accepté
    avertissement : 'created_at' (traçabilité) : aucune ligne de machine.sql:maintenance ne la fournit ; chargée vide (NULL) au Bronze, non reprise au Silver
    avertissement : 'updated_at' (traçabilité) : aucune ligne de machine.sql:maintenance ne la fournit ; chargée vide (NULL) au Bronze, non reprise au Silver
■ releves_incidents    accepté
■ telemetry            accepté

4 jeu(x) accepté(s), 0 refusé(s) : []


**Lecture du résultat.** Le guide `01-INGESTION_PROCESS.md` §3 (règle 8) cite ce cas précis :
*« `is_active` / `created_at` / `updated_at` présentes au `CREATE TABLE` mais absentes de l'`INSERT`
(leurs valeurs viendraient d'un `DEFAULT`, donc d'un élément qui n'est pas dans le fichier) »*.
Sans décision au contrat, ce contrôle **refuse** `machine` et `maintenance`.

✅ **Décision prise avec le formateur (2026-11-26)** : ce sont des **champs de traçabilité**, on les prend.
Les contrats (`machine` 2.1.0, `maintenance` 0.3.0) les déclarent avec `traceability: true` :
- au **Bronze**, la table est créée avec ces colonnes, chargées telles que la source les donne : vides
  (`NULL`), puisqu'aucun `INSERT` ne les fournit. La valeur du `DEFAULT` (`TRUE`, `NOW()`) n'est **pas**
  inventée. Le contrôle passe de l'erreur à l'**avertissement** ci-dessus ;
- au **Silver**, elles ne sont ni contrôlées ni reprises (opération `DROP` déclarée dans le dictionnaire,
  section 7).

Le contrôle, lui, ne s'assouplit pas : une colonne non alimentée **sans** `traceability: true` reste bloquante.

## 5. Étape 3 — Stocker brut, sans perte : `x_raw` + `x`

Le guide 02 propose deux doctrines ; le contrat v2 retient le **Bronze hybride** :

- `x_raw` (`TEXT`) : la valeur d'origine, **intacte** ;
- `x` : la valeur **convertie au mieux** vers le type du contrat (`BIGINT`, `DOUBLE PRECISION`,
  `TIMESTAMPTZ`, `TIME`, `BOOLEAN` ; `TEXT` pour identifiants et catégories).

Cela permet de distinguer les états d'une valeur (guide 02 §5) :

| Situation | `x_raw` | `x` | `_cast_errors` |
|---|---|---|---|
| présente et convertible | valeur d'origine | valeur typée | — |
| **absente** (case vide) | `''` | `NULL` | — (une absence n'est pas une erreur) |
| **illisible** (`n/a` dans une colonne numérique) | valeur d'origine | `NULL` | `{"x": "conversion … impossible : 'n/a'"}` |
| hors domaine (valeur impossible métier) | — | — | **pas au Bronze** : contrôlé au Silver |

Démonstration sur un petit jeu **fabriqué** (aucune donnée réelle) :

In [7]:
import uuid
from datetime import UTC, datetime

demo_meta = BatchMeta(uuid.uuid4(), "exemple.csv", "0" * 64, datetime.now(UTC))
demo = pd.DataFrame({"mesure": ["12.5", "", "n/a"], "compte": ["3", "4.5", "7"]}, dtype=str)
bf_demo = build_bronze_frame(demo, {"mesure": "FLOAT", "compte": "INTEGER"}, demo_meta, cfg)
bf_demo.frame[["mesure_raw", "mesure", "compte_raw", "compte", cfg.col("cast_errors")]]

,mesure_raw,mesure,compte_raw,compte,_cast_errors
0,12.5,12.5,3,3.0,{}
1,,None,4.5,None,{'compte': 'conversion INTEGER impossible : '4.5''}
2,n/a,None,7,7.0,{'mesure': 'conversion FLOAT impossible : 'n/a''}


In [8]:
print("Lignes avec au moins une erreur de conversion :", bf_demo.rows_with_cast_errors, "/", len(demo))
for i, errs in enumerate(bf_demo.frame[cfg.col("cast_errors")], start=1):
    print(f"  ligne {i} : {errs or '{}'}")

Lignes avec au moins une erreur de conversion : 2 / 3
  ligne 1 : {}
  ligne 2 : {'compte': "conversion INTEGER impossible : '4.5'"}
  ligne 3 : {'mesure': "conversion FLOAT impossible : 'n/a'"}


### Les colonnes techniques ajoutées

Chaque ligne porte son fichier, son empreinte, son numéro et son lot (guide 01 §4, étape 3).
Leurs noms sont dans la config (`layers.bronze.technical_columns`), pas dans le code.

| Colonne | Type SQL | Rôle | Demandé par |
|---|---|---|---|
| `_batch_id` | `UUID` | identifiant du lot d'ingestion (lien avec le registre) | guides 01 et Medallion |
| `_source_file` | `TEXT` | chemin du fichier d'origine (relatif au projet) | guides 01 et Medallion |
| `_source_hash` | `TEXT` | SHA-256 **des octets** du fichier | guides 01 et 02 (sceau : « quels octets ? ») |
| `_row_number` | `BIGINT` | numéro de l'enregistrement dans la source (1 = premier) | guide 01 |
| `_ingested_at` | `TIMESTAMPTZ` | horodatage du chargement, en UTC | guides 01 et Medallion |
| `_row_hash` | `TEXT` | SHA-256 des valeurs d'origine de la ligne (doublons stricts, traités au Silver) | guide Medallion (`_row_checksum`) |
| `_cast_errors` | `JSONB` | `{colonne: motif}` des conversions impossibles ; `{}` sinon | guide 01 |

**Clé primaire : `(_source_hash, _row_number)`** — l'unicité est portée par la base.

Construction **en mémoire** (sans écrire en base) de la table Bronze d'un jeu accepté. On prend
un jeu **sans colonne personnelle** pour qu'aucune donnée nominative n'apparaisse dans les sorties
du notebook (versionné dans Git).

In [9]:
def has_pii(ds):
    return any(x.get("pii") for x in contracts[ds.id]["schema"]["columns"])


ls0, ds0, prof0 = next((a for a in accepted if not has_pii(a[1])), accepted[0])
meta0 = BatchMeta(uuid.uuid4(), ls0.src.spec.path.name, ls0.src.sha256, datetime.now(UTC))
declared0 = {c["name"]: c.get("declared_type") for c in contracts[ds0.id]["schema"]["columns"]}
bf0 = build_bronze_frame(ds0.df, _types(ds0, contracts[ds0.id], prof0), meta0, cfg,
                         settings.inference.parse_rate_min, declared0)

print(f"Jeu : {ds0.id} — {ds0.df.shape[1]} colonnes source -> {bf0.frame.shape[1]} colonnes Bronze "
      f"(2 × {ds0.df.shape[1]} + {len(cfg.technical)} techniques)")
print("Ordre des colonnes =", physical_columns(list(ds0.df.columns), cfg) == list(bf0.frame.columns))
pd.DataFrame([{"colonne": c, "type PostgreSQL": (t() if isinstance(t, type) else t).compile(dialect=postgresql.dialect())}
              for c, t in bf0.sql_types.items()])

Jeu : machine — 11 colonnes source -> 29 colonnes Bronze (2 × 11 + 7 techniques)
Ordre des colonnes = True


,colonne,type PostgreSQL
0,machine_code_raw,TEXT
1,machine_code,TEXT
2,commissioning_date_raw,TEXT
3,commissioning_date,TIMESTAMP WITHOUT TIME ZONE
4,max_daily_capacity_raw,TEXT
5,max_daily_capacity,BIGINT
6,max_hourly_capacity_pieces_raw,TEXT
7,max_hourly_capacity_pieces,BIGINT
8,model_raw,TEXT
9,model,TEXT


In [10]:
bf0.frame.head(3)

,machine_code_raw,machine_code,commissioning_date_raw,commissioning_date,max_daily_capacity_raw,max_daily_capacity,max_hourly_capacity_pieces_raw,max_hourly_capacity_pieces,model_raw,model,production_line_raw,production_line,location_raw,location,criticality_raw,criticality,is_active_raw,is_active,created_at_raw,created_at,updated_at_raw,updated_at,_batch_id,_source_file,_source_hash,_row_number,_ingested_at,_row_hash,_cast_errors
0,MACH-01,MACH-01,2021-05-12,2021-05-12 00:00:00,770,770,48,48,InduPress-X2,InduPress-X2,Ligne-A,Ligne-A,Atelier-2,Atelier-2,MEDIUM,MEDIUM,,None,,None,,None,c6ba05ca-743a-4324-aed0-5c42f9a52902,machine.sql,9eb877ae806515e852df46dd9f97f31aabfa8420a8f07dc50a5be4cefced8fcb,1,2026-11-27 15:57:11.137939+00:00,92634b8239ebd6faa1a1f0a92a4166a369d62ce330f381e87997c3079ebfefa6,{}
1,MACH-02,MACH-02,2024-09-07,2024-09-07 00:00:00,800,800,50,50,InduPress-X2,InduPress-X2,Ligne-A,Ligne-A,Atelier-1,Atelier-1,LOW,LOW,,None,,None,,None,c6ba05ca-743a-4324-aed0-5c42f9a52902,machine.sql,9eb877ae806515e852df46dd9f97f31aabfa8420a8f07dc50a5be4cefced8fcb,2,2026-11-27 15:57:11.137939+00:00,f4c1434bb31c22c4de20c1f70a7d9f57b2fbf993dab370b6fdda3042286e8cc4,{}
2,MACH-03,MACH-03,2019-07-23,2019-07-23 00:00:00,1405,1405,88,88,InduPress-X1,InduPress-X1,Ligne-B,Ligne-B,Atelier-1,Atelier-1,HIGH,HIGH,,None,,None,,None,c6ba05ca-743a-4324-aed0-5c42f9a52902,machine.sql,9eb877ae806515e852df46dd9f97f31aabfa8420a8f07dc50a5be4cefced8fcb,3,2026-11-27 15:57:11.137939+00:00,a7791e077974f29cd899547a7c31080489f7e1f317f0d12138a9705ad82ac93e,{}


### Deux empreintes, deux usages

- `_source_hash` : identité du **fichier** (octets). Avec `_row_number`, c'est la clé primaire :
  le même fichier rechargé n'insère rien ; un fichier modifié est un **nouveau lot**.
- `_row_hash` : identité du **contenu** d'une ligne. Le Silver s'en sert pour retirer les doublons stricts.

In [11]:
print("['a,b', 'c'] ->", row_hash(["a,b", "c"])[:16], "…")
print("['a', 'b,c'] ->", row_hash(["a", "b,c"])[:16], "…   (différent : encodage JSON, pas de concaténation)")
dup = bf0.frame[cfg.col("row_hash")].duplicated().sum()
print(f"{ds0.id} : {dup} ligne(s) dont l'empreinte est déjà portée par une autre ligne "
      "(conservées au Bronze, traitées au Silver)")

['a,b', 'c'] -> 569c363fb7e855f8 …
['a', 'b,c'] -> 472ecd0a16762a33 …   (différent : encodage JSON, pas de concaténation)
machine : 0 ligne(s) dont l'empreinte est déjà portée par une autre ligne (conservées au Bronze, traitées au Silver)


## 6. Étape 4 — Seuil, transaction, registre des lots

1. **Seuil** : si la part de lignes portant au moins une erreur de conversion dépasse
   `max_cast_error_ratio` (`0.0` dans la config ; un contrat peut fixer le sien dans
   `source.max_cast_error_ratio`, décision à justifier), le lot est **annulé avant toute écriture**.
2. **Transaction** (`engine.begin()`) : `INSERT … ON CONFLICT DO NOTHING RETURNING` par paquets de
   `chunk_size` lignes. Tout le lot ou rien. Le chargement **ne crée aucune table** : la structure
   vient des migrations Alembic (`indusense db revision`, relue, puis `db upgrade` ; `docs/08` §4 bis).
   Table absente ou d'une autre structure : le lot est refusé, avec la commande à lancer.
3. **Registre** `bronze.ingestion_batches_log` : une ligne **par tentative** (réussie ou non), écrite
   dans **sa propre transaction** pour que l'échec reste tracé. On y trouve aussi l'empreinte du
   **contrat** (« quelle recette ? », sceau du guide 02).

In [12]:
registre = pd.read_sql_query(
    text(f'SELECT * FROM "{cfg.bronze_schema}"."{cfg.registry_table}" ORDER BY started_at DESC LIMIT 12'), engine)
registre["source_hash"] = registre["source_hash"].str[:12] + "…"
registre["contract_hash"] = registre["contract_hash"].str[:12] + "…"
registre[["started_at", "dataset_id", "status", "records_read", "records_inserted",
          "records_already_present", "rows_with_cast_errors", "contract_version", "source_hash",
          "contract_hash", "message"]]

,started_at,dataset_id,status,records_read,records_inserted,records_already_present,rows_with_cast_errors,contract_version,source_hash,contract_hash,message
0,2026-11-27 15:54:05.976831+00:00,telemetry,SUCCESS,135626,0,135626,0,0.3.0,68ec8dd09b69…,2b835b0a1948…,None
1,2026-11-27 15:54:04.395802+00:00,releves_incidents,SUCCESS,1245,0,1245,0,1.1.0,be7204b3320a…,42f7e85fcdbb…,None
2,2026-11-27 15:54:04.054646+00:00,maintenance,SUCCESS,115,0,115,0,0.5.0,9eb877ae8065…,625a17f2b88f…,None
3,2026-11-27 15:54:03.866040+00:00,machine,SUCCESS,15,0,15,0,2.2.0,9eb877ae8065…,d1533b87cbfb…,None
4,2026-11-27 15:50:53.801477+00:00,telemetry,SUCCESS,135626,0,135626,0,0.3.0,68ec8dd09b69…,2b835b0a1948…,None
5,2026-11-27 15:50:51.040720+00:00,releves_incidents,SUCCESS,1245,0,1245,0,1.1.0,be7204b3320a…,42f7e85fcdbb…,None
6,2026-11-27 15:50:50.768710+00:00,maintenance,SUCCESS,115,0,115,0,0.5.0,9eb877ae8065…,625a17f2b88f…,None
7,2026-11-27 15:50:50.599431+00:00,machine,SUCCESS,15,0,15,0,2.2.0,9eb877ae8065…,d1533b87cbfb…,None
8,2026-11-27 15:33:42.809298+00:00,telemetry,SUCCESS,135626,0,135626,0,0.2.0,68ec8dd09b69…,95994a53cca1…,None
9,2026-11-27 15:33:41.715028+00:00,releves_incidents,SUCCESS,1245,0,1245,0,1.0.0,be7204b3320a…,4c49a253fed8…,None


## 7. Étape 5 — Dictionnaire de données et section `transformations`

À chaque chargement, `artifacts/ingestion/layers/AAAAMMJJ/bronze_<jeu>_HHMMSS.md` et `.json` :
- une ligne par colonne : type, type SQL, vides, échecs de conversion, taux de NULL, min, max (**calculés**) ;
- `transformations` (obligatoire selon le guide 01) : **ce qui a été fait à la donnée** — `CAST`, `COPY`, `HASH`, `TECH` ;
- `silver_operations` : ce que le contrat confie au Silver — `DEL` (doublons), `ANO` (pseudonymisation),
  `DROP`, `TODO` (décision RGPD manquante).

In [13]:
print("Transformations faites au Bronze —", ds0.id)
display(pd.DataFrame(transformations(bf0, cfg)))
print("Opérations déclarées pour le Silver —", ds0.id)
display(pd.DataFrame(silver_operations(contracts[ds0.id], cfg)))

Transformations faites au Bronze — machine


,code,target,source,detail
0,COPY,machine_code,[machine_code_raw],copie de la valeur d'origine (case vide -> NULL)
1,CAST,commissioning_date,[commissioning_date_raw],conversion au mieux vers DATETIME (DATETIME) ; valeur d'origine conservée da...
2,CAST,max_daily_capacity,[max_daily_capacity_raw],conversion au mieux vers INTEGER (BIGINT) ; valeur d'origine conservée dans ...
3,CAST,max_hourly_capacity_pieces,[max_hourly_capacity_pieces_raw],conversion au mieux vers INTEGER (BIGINT) ; valeur d'origine conservée dans ...
4,COPY,model,[model_raw],copie de la valeur d'origine (case vide -> NULL)
5,COPY,production_line,[production_line_raw],copie de la valeur d'origine (case vide -> NULL)
6,COPY,location,[location_raw],copie de la valeur d'origine (case vide -> NULL)
7,COPY,criticality,[criticality_raw],copie de la valeur d'origine (case vide -> NULL)
8,CAST,is_active,[is_active_raw],conversion au mieux vers BOOLEAN (BOOLEAN) ; valeur d'origine conservée dans...
9,CAST,created_at,[created_at_raw],conversion au mieux vers DATETIME (DATETIME) ; valeur d'origine conservée da...


Opérations déclarées pour le Silver — machine


,code,target,source,detail
0,DEL,lignes,[_row_hash],doublons stricts (même empreinte de ligne) : une seule ligne gardée
1,DROP,is_active,[is_active],"colonne de traçabilité (contrat : traceability) : gardée au Bronze, non repr..."
2,DROP,created_at,[created_at],"colonne de traçabilité (contrat : traceability) : gardée au Bronze, non repr..."
3,DROP,updated_at,[updated_at],"colonne de traçabilité (contrat : traceability) : gardée au Bronze, non repr..."


In [14]:
# Dernier dictionnaire écrit en base pour chaque jeu chargé
layers_dir = settings.artifact_dir("layers")
for ls, ds, _ in accepted:
    files = sorted(layers_dir.glob(f"*/bronze_{ds.id}_*.md"))
    print(f"{ds.id:<20} -> {files[-1].relative_to(ROOT) if files else 'aucun dictionnaire'}")
files0 = sorted(layers_dir.glob(f"*/bronze_{ds0.id}_*.md"))
if files0:
    display(Markdown(files0[-1].read_text(encoding="utf-8")))

machine              -> artifacts\ingestion\layers\20261127\bronze_machine_165356.md
maintenance          -> artifacts\ingestion\layers\20261127\bronze_maintenance_165356.md
releves_incidents    -> artifacts\ingestion\layers\20261127\bronze_releves_incidents_165356.md
telemetry            -> artifacts\ingestion\layers\20261127\bronze_telemetry_165356.md


# Bronze — machine

- Statut du lot : **SUCCESS**
- Lot : `7d4e3139-9e8e-4347-876b-c074ddc1be2a`
- Source : `datas/machine.sql` (SHA-256 `9eb877ae806515e852df46dd9f97f31aabfa8420a8f07dc50a5be4cefced8fcb`)
- Contrat : version 2.2.0 (SHA-256 `d1533b87cbfbed3eaf444a64b33da2fce37566038db49fc92e3fed899cdf53ef`)
- Lignes lues : 15 ; insérées : 0 ; déjà présentes : 15
- Lignes avec au moins une erreur de conversion : 0 (seuil : 0.00%)

## Colonnes

| Source | Brute | Typée | Type | Type SQL | Vides | Échecs conv. | Taux NULL | Min | Max |
|---|---|---|---|---|---|---|---|---|---|
| machine_code | machine_code_raw | machine_code | IDENTITY | TEXT | 0 | 0 | 0.00% |  |  |
| commissioning_date | commissioning_date_raw | commissioning_date | DATETIME | DATETIME | 0 | 0 | 0.00% | 2019-07-23 00:00:00 | 2025-05-25 00:00:00 |
| max_daily_capacity | max_daily_capacity_raw | max_daily_capacity | INTEGER | BIGINT | 0 | 0 | 0.00% | 750 | 1428 |
| max_hourly_capacity_pieces | max_hourly_capacity_pieces_raw | max_hourly_capacity_pieces | INTEGER | BIGINT | 0 | 0 | 0.00% | 47 | 89 |
| model | model_raw | model | CATEGORY | TEXT | 0 | 0 | 0.00% |  |  |
| production_line | production_line_raw | production_line | CATEGORY | TEXT | 0 | 0 | 0.00% |  |  |
| location | location_raw | location | CATEGORY | TEXT | 0 | 0 | 0.00% |  |  |
| criticality | criticality_raw | criticality | CATEGORY | TEXT | 0 | 0 | 0.00% |  |  |
| is_active | is_active_raw | is_active | BOOLEAN | BOOLEAN | 15 | 0 | 100.00% |  |  |
| created_at | created_at_raw | created_at | DATETIME | DATETIME | 15 | 0 | 100.00% |  |  |
| updated_at | updated_at_raw | updated_at | DATETIME | DATETIME | 15 | 0 | 100.00% |  |  |

## Transformations faites au Bronze

| Code | Cible | Source | Détail |
|---|---|---|---|
| COPY | machine_code | machine_code_raw | copie de la valeur d'origine (case vide -> NULL) |
| CAST | commissioning_date | commissioning_date_raw | conversion au mieux vers DATETIME (DATETIME) ; valeur d'origine conservée dans commissioning_date_raw ; case vide -> NULL sans erreur ; échec -> NULL et motif dans _cast_errors |
| CAST | max_daily_capacity | max_daily_capacity_raw | conversion au mieux vers INTEGER (BIGINT) ; valeur d'origine conservée dans max_daily_capacity_raw ; case vide -> NULL sans erreur ; échec -> NULL et motif dans _cast_errors |
| CAST | max_hourly_capacity_pieces | max_hourly_capacity_pieces_raw | conversion au mieux vers INTEGER (BIGINT) ; valeur d'origine conservée dans max_hourly_capacity_pieces_raw ; case vide -> NULL sans erreur ; échec -> NULL et motif dans _cast_errors |
| COPY | model | model_raw | copie de la valeur d'origine (case vide -> NULL) |
| COPY | production_line | production_line_raw | copie de la valeur d'origine (case vide -> NULL) |
| COPY | location | location_raw | copie de la valeur d'origine (case vide -> NULL) |
| COPY | criticality | criticality_raw | copie de la valeur d'origine (case vide -> NULL) |
| CAST | is_active | is_active_raw | conversion au mieux vers BOOLEAN (BOOLEAN) ; valeur d'origine conservée dans is_active_raw ; case vide -> NULL sans erreur ; échec -> NULL et motif dans _cast_errors |
| CAST | created_at | created_at_raw | conversion au mieux vers DATETIME (DATETIME) ; valeur d'origine conservée dans created_at_raw ; case vide -> NULL sans erreur ; échec -> NULL et motif dans _cast_errors |
| CAST | updated_at | updated_at_raw | conversion au mieux vers DATETIME (DATETIME) ; valeur d'origine conservée dans updated_at_raw ; case vide -> NULL sans erreur ; échec -> NULL et motif dans _cast_errors |
| HASH | _row_hash | machine_code_raw, commissioning_date_raw, max_daily_capacity_raw, max_hourly_capacity_pieces_raw, model_raw, production_line_raw, location_raw, criticality_raw, is_active_raw, created_at_raw, updated_at_raw | SHA-256 de la liste JSON des valeurs d'origine de la ligne |
| TECH | _batch_id, _source_file, _source_hash, _row_number, _ingested_at |  | colonnes de traçabilité ajoutées ; aucune valeur source modifiée |

## Opérations déclarées pour le Silver

| Code | Cible | Source | Détail |
|---|---|---|---|
| DEL | lignes | _row_hash | doublons stricts (même empreinte de ligne) : une seule ligne gardée |
| DROP | is_active | is_active | colonne de traçabilité (contrat : traceability) : gardée au Bronze, non reprise au Silver |
| DROP | created_at | created_at | colonne de traçabilité (contrat : traceability) : gardée au Bronze, non reprise au Silver |
| DROP | updated_at | updated_at | colonne de traçabilité (contrat : traceability) : gardée au Bronze, non reprise au Silver |


## 8. Étape 6 — Vérifier les invariants en base

Le guide 02 demande de tester des **propriétés de la donnée**, pas du code. On les vérifie ici
directement sur les tables `bronze.*`, pour le lot du fichier actuel (`_source_hash`) :

| Invariant | Vérification |
|---|---|
| Volume | lignes en base = lignes lues dans le fichier |
| Fidélité | chaque `x_raw` est identique à la source, valeur pour valeur, dans l'ordre de `_row_number` |
| Complétude | `x` NULL ⟺ case vide **ou** motif dans `_cast_errors` |
| Unicité | aucun couple `(_source_hash, _row_number)` en double |

In [15]:
def q(sql, **params):
    return pd.read_sql_query(text(sql), engine, params=params)


tables = set(list_tables(engine, cfg.bronze_schema))
B = cfg.bronze_schema
H, N, E = cfg.col("source_hash"), cfg.col("row_number"), cfg.col("cast_errors")
rows = []
for ls, ds, _ in accepted:
    if ds.id not in tables:
        rows.append({"jeu": ds.id, "statut": "table absente : lancer `indusense load bronze`"})
        continue
    h, cols = ls.src.sha256, list(ds.df.columns)
    raw_cols = ", ".join(f'"{c}{cfg.raw_suffix}"' for c in cols)
    base = q(f'SELECT {raw_cols} FROM "{B}"."{ds.id}" WHERE "{H}" = :h ORDER BY "{N}"', h=h)
    fidele = base.fillna("").to_numpy().tolist() == ds.df.to_numpy().tolist()
    incoherents = sum(q(f'''SELECT count(*) AS n FROM "{B}"."{ds.id}" WHERE "{H}" = :h AND
                ("{c}" IS NULL) <> (trim(coalesce("{c}{cfg.raw_suffix}", '')) = '' OR jsonb_exists("{E}", :c))''',
                        h=h, c=c)["n"][0] for c in cols)
    doublons = q(f'SELECT count(*) AS n FROM (SELECT 1 FROM "{B}"."{ds.id}" GROUP BY "{H}", "{N}" '
                 'HAVING count(*) > 1) d')["n"][0]
    rows.append({"jeu": ds.id, "lignes fichier": len(ds.df), "lignes en base (ce fichier)": len(base),
                 "volume OK": len(base) == len(ds.df), "fidélité x_raw": fidele,
                 "NULL incohérents": int(incoherents), "clés en double": int(doublons)})
pd.DataFrame(rows)

,jeu,lignes fichier,lignes en base (ce fichier),volume OK,fidélité x_raw,NULL incohérents,clés en double
0,machine,15,15,True,True,0,0
1,maintenance,115,115,True,True,0,0
2,releves_incidents,1245,1245,True,True,0,0
3,telemetry,135626,135626,True,True,0,0


## 9. Étape 7 — Sceller et rendre rejouable : le rejeu

L'idempotence est portée par une **contrainte de base** (clé primaire) et `ON CONFLICT DO NOTHING`,
pas par un `if deja_vu` dans le code (guide 02 §3 : *« un `if` se contourne, une clé primaire non »*).

On relance le chargement Bronze tel quel : attendu **0 ligne insérée**, tout « déjà présent »,
et une nouvelle ligne `SUCCESS` au registre. (Cette cellule écrit dans la base et dans `artifacts/`,
exactement comme `uv run indusense load bronze`.)

In [16]:
REJOUER = True
if REJOUER:
    run = run_layers(settings, ["bronze"], env=env, engine=engine)
    display(pd.DataFrame(run.summary.get("bronze", {})).T)
    print("Refusés :", run.summary.get("refused") or "aucun")

INFO    | Context impl PostgresqlImpl.


INFO    | Will assume transactional DDL.


INFO    | lecture de machine.sql


INFO    | lecture de releves_incidents.csv


INFO    | lecture de telemetry.csv


WARNING | machine : 'is_active' (traçabilité) : aucune ligne de datas/machine.sql:machine ne la fournit ; chargée vide (NULL) au Bronze, non reprise au Silver


... (15 outputs left out)

INFO    | bronze.telemetry : 135626 lues, 0 insérées, 135626 déjà présentes, 0 avec erreur de conversion


INFO    | manifeste : C:\dev\00-Fomation_IA\TP1\indusense\artifacts\ingestion\runs\20261127_165713\manifest.json


,status,records_read,records_inserted,records_already_present,rows_with_cast_errors,batch_id
machine,SUCCESS,15,0,15,0,f0ff6bf7-0ec4-4b1f-9f09-ffc86367a703
maintenance,SUCCESS,115,0,115,0,66cbaa1e-0687-4463-8cca-e48c1a917fe9
releves_incidents,SUCCESS,1245,0,1245,0,57ea1812-e820-4970-9bd9-bcc8617118fa
telemetry,SUCCESS,135626,0,135626,0,b294bdae-42bf-42fd-851c-c17a66c1881d


Refusés : aucun


## 10. Volet réglementaire (RGPD) — ce que fait le Bronze, et ce qu'il ne fait pas

Le Bronze **ne transforme rien** : les noms, badges et commentaires y sont **en clair**, comme dans le
fichier source. La protection a lieu au **Silver** (opération `ANO` : HMAC-SHA256 + sel secret du
`.env`). Rappel du document `13_RGPD` : un pseudonyme **reste une donnée personnelle** tant que le
lien peut être rétabli (le détenteur du sel le peut).

| Principe (RGPD art. 5) | Ce que le dépôt fait déjà | Reste à faire valider / faire |
|---|---|---|
| 1. Licéité, transparence | colonnes `pii: true` inventoriées dans les contrats | base légale et information des salariés : responsable du traitement |
| 2. Limitation des finalités | finalité décrite dans chaque contrat | fiche de finalité par jeu de données |
| 3. Minimisation | décision par colonne (`privacy: drop / pseudonymize / to_decide`), opérations déclarées dans le dictionnaire | trancher `to_decide` (texte libre) ; exclure le commentaire des features et exports |
| 4. Exactitude | aucune correction silencieuse ; échecs tracés (`_cast_errors`) ; lignage Silver → Bronze → fichier | historique des corrections (Silver/Gold) |
| 5. Conservation | `_ingested_at`, `_batch_id`, registre | **durée de conservation du Bronze en clair** et purge : aucune à ce jour |
| 6. Intégrité, confidentialité | base sur `127.0.0.1`, secrets dans `.env` (ignoré par Git), aucune valeur personnelle dans les logs | **habilitations** : accès au schéma `bronze` réservé à qui charge les données ; sel réel hors dépôt |
| 7. Responsabilité | empreintes fichier + contrat, registre, dictionnaires, manifestes, tests | fiche dataset signée, matrice d'habilitation |

Le guide Medallion (§3.1) ajoute l'angle **IA Act / non-discrimination** : le modèle qualifie la santé
des machines, pas le rendement d'un opérateur — d'où l'exclusion des identifiants personnels des features.

In [17]:
# Inventaire des colonnes personnelles : décision et présence en clair au Bronze (on compte, on n'affiche AUCUNE valeur)
silver_tables = set(list_tables(engine, cfg.silver_schema))
rows = []
for ls, ds, _ in accepted:
    c = contracts[ds.id]
    to_protect = columns_to_pseudonymize(c, cfg.pseudonymize_privacy)
    for col in [x for x in c["schema"]["columns"] if x.get("pii")]:
        name = col["name"]
        en_clair = q(f'''SELECT count(*) AS n FROM "{B}"."{ds.id}" WHERE trim("{name}{cfg.raw_suffix}") <> ''
                    ''')["n"][0] if ds.id in tables else None
        silver = "table absente"
        if name not in to_protect:
            silver = ("en clair (décision : keep)" if col.get("privacy") == "keep"
                      else "EN CLAIR (aucune protection)")
        elif ds.id in silver_tables:
            cols_silver = q("SELECT column_name FROM information_schema.columns WHERE table_schema = :s "
                            "AND table_name = :t", s=cfg.silver_schema, t=ds.id)["column_name"].tolist()
            if name not in cols_silver:
                silver = "colonne retirée"
            else:
                n_hash = q(f'''SELECT count(*) AS n FROM "{cfg.silver_schema}"."{ds.id}"
                             WHERE "{name}" ~ '^[0-9a-f]{{64}}$' ''')["n"][0]
                silver = f"{n_hash} valeurs pseudonymisées (64 hex)"
        rows.append({"jeu": ds.id, "colonne": name, "privacy": col.get("privacy"),
                     "protégée au Silver": name in to_protect,
                     "valeurs non vides en clair au Bronze": en_clair, "état au Silver": silver})
display(pd.DataFrame(rows))

for ls, ds, _ in accepted:
    undecided = undecided_pii_columns(contracts[ds.id])
    if undecided:
        print(f"⚠ {ds.id} : donnée personnelle SANS décision de protection -> {undecided}")
print(".env ignoré par Git :", ".env" in (ROOT / ".gitignore").read_text(encoding="utf-8").split())
print("Sel d'exemple utilisé :", env.anonymisation_salt == "dev-salt-indusense-change-me",
      "(acceptable en formation, jamais en réel)")

,jeu,colonne,privacy,protégée au Silver,valeurs non vides en clair au Bronze,état au Silver
0,releves_incidents,operator_name,drop,True,1245,colonne retirée
1,releves_incidents,operator_badge,pseudonymize,True,1245,1245 valeurs pseudonymisées (64 hex)
2,releves_incidents,comment,keep,False,1245,en clair (décision : keep)


.env ignoré par Git : True
Sel d'exemple utilisé : True (acceptable en formation, jamais en réel)


## 11. Écarts avec les consignes du formateur

On confronte ce qui est implémenté aux consignes (`01-INGESTION_PROCESS.md`, checklist §11 du
guide 02, guide Medallion §2). La cellule ci-dessous **mesure** ce qui peut l'être ; le tableau qui
suit liste tous les écarts.

In [18]:
NUMERIC = {"INTEGER", "FLOAT"}
reg_cols = q("SELECT column_name FROM information_schema.columns WHERE table_schema = :s AND table_name = :t",
             s=B, t=cfg.registry_table)["column_name"].tolist()
checks = []
for cid, c in contracts.items():
    num = [x["name"] for x in c["schema"]["columns"] if x.get("type") in NUMERIC]
    checks.append({"contrat": cid,
                   "horloge déclarée (timezone)": "timezone" in c or "timezone" in (c.get("source") or {}),
                   "colonnes numériques sans unité": [n for n in num if not next(
                       x for x in c["schema"]["columns"] if x["name"] == n).get("unit")],
                   "encodage du contrat": (c.get("source") or {}).get("encoding", "non déclaré"),
                   "dérivées déclarées (MERGE…)": bool(c.get("derived_columns"))})
display(pd.DataFrame(checks))
spec_enc = {s.id: s.encoding for s in specs}
print("Encodage réellement utilisé par la lecture :", spec_enc)
applied, heads = schema_status(engine, settings, cfg)
print("Migrations Alembic — révision appliquée :", sorted(applied), "; dernière :", sorted(heads),
      "; à jour :", applied == heads)
print("Registre — colonnes du guide Medallion absentes :",
      [c for c in ["source_name", "source_format", "records_rejected"] if c not in reg_cols])
print("Tests d'intégration PostgreSQL activés (INDUSENSE_TEST_DB_URL) :",
      bool(__import__("os").environ.get("INDUSENSE_TEST_DB_URL")))

,contrat,horloge déclarée (timezone),colonnes numériques sans unité,encodage du contrat,dérivées déclarées (MERGE…)
0,machine,False,[],non déclaré,False
1,maintenance,False,[],non déclaré,False
2,releves_incidents,False,[],utf-8,False
3,telemetry,False,[],utf-8,False


INFO    | Context impl PostgresqlImpl.


INFO    | Will assume transactional DDL.


Encodage réellement utilisé par la lecture : {'machine': 'auto', 'releves_incidents': 'auto', 'telemetry': 'auto'}
Migrations Alembic — révision appliquée : ['d027937788a3'] ; dernière : ['d027937788a3'] ; à jour : True
Registre — colonnes du guide Medallion absentes : ['source_name', 'source_format', 'records_rejected']
Tests d'intégration PostgreSQL activés (INDUSENSE_TEST_DB_URL) : False


| # | Consigne (source) | État dans le dépôt | Proposition |
|---|---|---|---|
| 1 | Contrat confronté au fichier réel, règle 8 (guide 01 §3) | ✅ implémentée ; colonnes `DEFAULT` de `machine` / `maintenance` déclarées `traceability: true` (décision formateur, section 4) : chargées vides au Bronze, non reprises au Silver | — |
| 2 | `x_raw` + `x`, colonnes techniques, `_cast_errors` (guide 01 §4) | ✅ | — |
| 3 | Clé primaire `(hash, ligne)` + rejeu sans doublon (guides 01, 02) | ✅ vérifié section 9 | — |
| 4 | Registre des lots, transaction séparée (guides 01, 02) | ✅ ; colonnes différentes du modèle Medallion (`records_already_present` au lieu de `records_rejected`, pas de `source_format`, pas de statut `PARTIAL` — le lot est tout ou rien) | écart à valider avec le formateur |
| 5 | Section `transformations` obligatoire (guide 01 §4, étape 6) | ✅ `CAST / COPY / HASH / TECH` ; opérations Silver `DEL / ANO / DROP / TODO` (dont `DROP` des champs de traçabilité) | — |
| 6 | **Horloge déclarée `timezone: UTC`, étiquetée jamais convertie** (guide 01 §2) | ❌ absente des contrats | ajouter la clé au contrat + règle + test |
| 7 | **Unité obligatoire pour les colonnes numériques** (guide 01 §2, checklist §11) | ⚠ toutes renseignées aujourd'hui (tableau mesuré), mais `contract check` ne la rend pas obligatoire | ajouter la règle + un test |
| 8 | **Encodage explicite**, auto-détection supprimée (guide 01 §2) | ⚠ déclaré dans les contrats CSV mais la lecture utilise la détection automatique de `pipeline.yaml` | lire l'encodage du contrat, ou l'imposer par motif dans `discovery.overrides` |
| 9 | **Clé métier explicite** `natural_key` (guide 01 §2) | ✅ équivalent : `primary_key` du contrat | — |
| 10 | **`derived_columns`** déclarées au Bronze (fusion `date` + `time` = `MERGE`) (guide 01 §2) | ❌ le dictionnaire ne déclare que `DEL / ANO / DROP / TODO` | déclarer les dérivées dans le contrat, exécutées au Silver |
| 11 | **Migrations Alembic** revues (guide 01 §4, étape 4) | ✅ `indusense db revision` (généré depuis contrats + fichiers, destructions signalées) → relecture → `db upgrade` ; chargement refusé si la base est en retard ; `--rebuild` supprimé | — |
| 12 | Lecture **en flux** (mémoire indépendante de la taille) (guide 01 §4, étape 5) | ❌ le fichier est lu entièrement par pandas | lecture par morceaux si les volumes grossissent |
| 13 | Le sceau : « ce couple (fichier, configuration) a-t-il déjà réussi ? » (guide 02 §3) | ⚠ l'unicité porte sur le fichier ; l'empreinte du contrat est au registre mais ne fait pas partie de la clé | à discuter : recharger si seul le contrat change ? |
| 14 | Tests d'intégration PostgreSQL (guide 01 §8) | ⚠ 5 tests ignorés sans `INDUSENSE_TEST_DB_URL` (dont 2 sur les migrations) | définir la variable et les lancer |
| 15 | Graphes Bronze : complétude d'ingestion, latence, disponibilité du flux (guide Medallion §2.2) | ❌ aucun | à ajouter : `matplotlib` et `seaborn` installés (Q-08) |
| 16 | Habilitations sur le schéma `bronze` (données en clair) (RGPD principe 6) | ❌ un seul utilisateur de base | rôle dédié en lecture restreinte |
| 17 | Durée de conservation du Bronze en clair (RGPD principe 5) | ❌ non définie | à faire valider par le responsable du traitement |
| 18 | Volume `telemetry` : 135 626 lignes (guide 01 §2) | mesuré en section 3 ; `docs/08` indique 135 630 | corriger la documentation si le fichier n'a pas changé |
| 19 | **Indice de confiance des manquants par ligne** (guide 14 didactique, §1) | ❌ absent du Bronze | colonne technique décidée (Q-04), étape 1 du plan |

## 12. Synthèse

**Ce que la méthode garantit aujourd'hui** (vérifié dans ce notebook) :
- rien n'est écrit si le contrat et le fichier ne concordent pas (erreur localisée, colonne + fichier) ;
- la valeur d'origine est toujours consultable (`x_raw`), la valeur typée est à côté (`x`), et un échec
  de conversion est distingué d'une absence (`_cast_errors`) ;
- chaque ligne est traçable jusqu'à son fichier, son lot et sa position ;
- rejouer n'insère rien : l'unicité est garantie par la base ;
- chaque tentative est inscrite au registre, avec l'empreinte du fichier et celle du contrat ;
- un dictionnaire de données dit ce qui a été fait à la donnée et ce qui reste à faire au Silver.

**Décisions prises** (détail dans `docs/10_registre_questions_metier.md`) :
- colonnes alimentées par un `DEFAULT` (`machine` / `maintenance`) : champs de traçabilité, gardés au
  Bronze (vides), non repris au Silver — décision du formateur, 2026-11-26 (section 4) ;
- `maintenance.maintenance_type` (Q-03) : la liste du contrat (`preventive`, `corrective`) ne correspondait pas
  à la contrainte de la source (`CHECK (maintenance_type IN ('proactive','reactive'))`) ; contrat corrigé en
  0.4.0 — 2026-11-27 ;
- indice de confiance des manquants (Q-04) : **colonne technique de chaque ligne du Bronze**, à implémenter à
  l'étape 1 du plan (`docs/09_plan_bronze_vers_silver.md`) ; définition à fixer (Q-19) — 2026-11-27.

**Décisions à prendre** (elles ne relèvent pas du code) :
1. définition de l'indice de confiance par ligne (Q-19) : colonnes prises en compte, jetons d'absence,
   valeurs non convertibles ;
2. horloge des horodatages (section 11, n° 6) — **à confirmer avec le métier** (les horodatages sont aujourd'hui sans fuseau) ;
3. décision RGPD sur le texte libre (`privacy: to_decide`), durée de conservation et habilitations du Bronze (section 10).